# 📚 04: 기본 RAG - 검색 + LLM 연결

---

| 항목 | 내용 |
|------|------|
| **목표** | 검색 결과를 LLM 답변에 연결하는 기본 RAG 구조를 직접 구현하고 효과를 확인한다 |
| **예상 실행 시간** | ⏱️ 빠른 시연 5분 / 전체 15분 |
| **API 키** | ✅ 권장 (없으면 mock LLM 응답으로 흐름 시연) |
| **이전 노트북과의 연결** | 임베딩으로 문서를 검색할 수 있게 됐다. 이제 그 검색 결과를 LLM 답변에 연결하는 것이 RAG다. |

---

## 🎯 핵심 메시지

> **RAG = Retrieval-Augmented Generation**  
> **모델 재학습이 아니라, 답변 직전에 지식을 붙여주는 패턴입니다.**

```
사용자 질문
    ↓
관련 문서 검색 (Retrieval)
    ↓
질문 + 검색 결과 → LLM 프롬프트 (Augmented)
    ↓
LLM이 문서를 참고하여 답변 (Generation)
```

**LLM 단독 vs RAG 비교**가 이 노트북의 핵심 시연입니다.

In [ ]:
!pip install -q openai sentence-transformers rank-bm25
print("✅ 완료")

## 1️⃣ 환경 설정 + 벡터 인덱스 구축

In [ ]:
import os, json
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# .env 에서 API 키 로드 (python-dotenv 필요, Colab 에서는 직접 입력 가능)
try:
    from dotenv import load_dotenv
    from pathlib import Path
    for _p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (_p / ".env").exists():
            load_dotenv(_p / ".env"); break
except ImportError:
    pass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

def setup(api_key=""):
    key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if key and key not in ("", "sk-..."):
        try:
            from openai import OpenAI
            c = OpenAI(api_key=key)
            print("✅ API 모드")
            return "api", c
        except: pass
    print("💡 로컬 모드 - mock LLM + 로컬 임베딩")
    return "local", None

MODE, client = setup(OPENAI_API_KEY)

_st_model = None
def embed_texts(texts):
    global _st_model
    if MODE == "api" and client:
        try:
            vecs = []
            for i in range(0, len(texts), 50):
                resp = client.embeddings.create(input=texts[i:i+50], model="text-embedding-3-small")
                vecs.extend([r.embedding for r in resp.data])
            return np.array(vecs, dtype=np.float32)
        except: pass
    if _st_model is None:
        print("📥 임베딩 모델 로딩...")
        from sentence_transformers import SentenceTransformer
        _st_model = SentenceTransformer("all-MiniLM-L6-v2")
        print("✅")
    return _st_model.encode(texts, convert_to_numpy=True, show_progress_bar=False).astype(np.float32)

def cosine_sim(q, D):
    q = np.array(q, dtype=np.float32).flatten()
    D = np.array(D, dtype=np.float32)
    qn = np.linalg.norm(q)
    if qn < 1e-9: return np.zeros(len(D))
    dn = np.linalg.norm(D, axis=1)
    dn = np.where(dn < 1e-9, 1e-9, dn)
    return (D @ q) / (dn * qn)

def call_llm(prompt, system="당신은 테크코어 내부 지식 어시스턴트입니다.",
             mock=None, temperature=0.3):
    if MODE == "api" and client:
        try:
            r = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role":"system","content":system},{"role":"user","content":prompt}],
                temperature=temperature, max_tokens=600
            )
            return r.choices[0].message.content.strip()
        except Exception as e:
            print(f"⚠️ {e}")
    return f"[Mock]\n{mock}" if mock else "[로컬 모드]"

print(f"모드: {MODE}")

In [ ]:
import sys
from pathlib import Path


def _ensure_project_root_on_path() -> None:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    try:
        candidates.extend(path for path in cwd.iterdir() if path.is_dir())
    except OSError:
        pass

    for candidate in candidates:
        if (candidate / "helpers" / "sample_data.py").exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return

    raise ModuleNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. `ai_special_course` 저장소를 clone 한 뒤 "
        "repo root 또는 notebooks 디렉터리에서 노트북을 실행하세요."
    )


_ensure_project_root_on_path()

# 문서 데이터 + 벡터 인덱스 구축
from helpers.sample_data import SAMPLE_DOCS_MINI_04 as SAMPLE_DOCS
# Colab 사용 시: !git clone <repo> 후 sys.path 에 추가하거나, sample_data.py 를 /content 에 업로드하세요.

# 벡터 인덱스 구축
print("📊 벡터 인덱스 구축 중...")
doc_contents = [d["content"] for d in SAMPLE_DOCS]
doc_vecs = embed_texts(doc_contents)
print(f"✅ 완료! {len(SAMPLE_DOCS)}개 문서, {doc_vecs.shape[1]}차원 벡터")

def vector_search(query, top_k=3):
    q_vec = embed_texts([query])[0]
    scores = cosine_sim(q_vec, doc_vecs)
    idx = np.argsort(scores)[::-1][:top_k]
    return [{**SAMPLE_DOCS[i], "score": float(scores[i])} for i in idx]

## 2️⃣ RAG 파이프라인 구현

In [ ]:
def build_rag_prompt(query, docs, max_chars=2500):
    """검색된 문서로 RAG 프롬프트를 구성합니다."""
    context_parts = []
    total = 0
    for i, doc in enumerate(docs, 1):
        chunk = f"[참고 문서 {i}: {doc['doc_id']}]\n{doc['content']}"
        if total + len(chunk) > max_chars:
            break
        context_parts.append(chunk)
        total += len(chunk)

    context = "\n\n---\n\n".join(context_parts)

    return f"""다음 내부 문서를 참고하여 질문에 정확하게 답변해주세요.
문서에 없는 내용은 "해당 정보를 찾을 수 없습니다"라고 솔직하게 답하세요.
답변 마지막에 어떤 문서를 참고했는지 [참고: 문서ID] 형식으로 명시하세요.

=== 참고 문서 ===
{context}

=== 질문 ===
{query}

=== 답변 ==="""


def rag_pipeline(query, top_k=3, verbose=True):
    """전체 RAG 파이프라인을 실행합니다."""
    if verbose:
        print(f"\n{'='*55}")
        print(f"  RAG 파이프라인 실행")
        print(f"  쿼리: '{query}'")
        print(f"{'='*55}")

    # Step 1: Retrieval
    retrieved = vector_search(query, top_k=top_k)
    if verbose:
        print(f"\n[Step 1] 관련 문서 검색 완료:")
        for i, r in enumerate(retrieved, 1):
            print(f"  #{i} [{r['doc_id']}] 유사도: {r['score']:.4f}")

    # Step 2: Augment
    rag_prompt = build_rag_prompt(query, retrieved)
    if verbose:
        print(f"\n[Step 2] RAG 프롬프트 구성 완료 ({len(rag_prompt)}자)")

    # Step 3: Generate
    answer = call_llm(rag_prompt)
    if verbose:
        print(f"\n[Step 3] LLM 답변 생성 완료")

    return {
        "query": query,
        "retrieved_docs": retrieved,
        "answer": answer,
    }

print("✅ RAG 파이프라인 정의 완료")

## 3️⃣ 핵심 비교: LLM 단독 vs RAG

같은 질문에 대해 LLM 단독 답변과 RAG 답변을 나란히 비교합니다.  
**답변 품질 차이가 명확히 드러나는 질문**을 선택했습니다.

In [ ]:
import sys
from pathlib import Path


def _ensure_project_root_on_path() -> None:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    try:
        candidates.extend(path for path in cwd.iterdir() if path.is_dir())
    except OSError:
        pass

    for candidate in candidates:
        if (candidate / "helpers" / "sample_data.py").exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return

    raise ModuleNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. `ai_special_course` 저장소를 clone 한 뒤 "
        "repo root 또는 notebooks 디렉터리에서 노트북을 실행하세요."
    )


_ensure_project_root_on_path()

# Mock 응답들 (API 없는 경우 사용)
from helpers.sample_data import MOCK_LLM_ONLY, MOCK_RAG_RESPONSE
# Colab 사용 시: !git clone <repo> 후 sys.path 에 추가하거나, sample_data.py 를 /content 에 업로드하세요.

print("✅ Mock 응답 준비 완료")

In [ ]:
def compare_llm_vs_rag(query, mock_key, show_sources=True):
    """LLM 단독 vs RAG 결과를 HTML로 비교 출력합니다."""
    print(f"\n🔍 질문: '{query}'")
    print("─" * 55)

    # LLM 단독
    llm_only_prompt = f"""질문에 답변해주세요: {query}"""
    llm_answer = call_llm(llm_only_prompt, mock=MOCK_LLM_ONLY.get(mock_key, "[LLM 단독 응답]"))

    # RAG
    retrieved = vector_search(query, top_k=3)
    rag_prompt = build_rag_prompt(query, retrieved)
    rag_answer = call_llm(rag_prompt, mock=MOCK_RAG_RESPONSE.get(mock_key, "[RAG 응답]"))

    # HTML 비교 출력
    llm_html = llm_answer.replace('\n', '<br>').replace('  ', '&nbsp;&nbsp;')
    rag_html = rag_answer.replace('\n', '<br>').replace('  ', '&nbsp;&nbsp;')

    sources_html = ""
    if show_sources:
        src_items = "".join(
            f"<li>📄 <strong>{r['doc_id']}</strong> (유사도: {r['score']:.4f}) - {r['content'][:60]}...</li>"
            for r in retrieved
        )
        sources_html = f"""
        <div style="padding:12px;background:#f8f9fa;border-top:1px solid #ddd;">
          <strong>📚 RAG에서 사용한 문서:</strong>
          <ul style="font-size:12px;margin:6px 0;padding-left:18px;">{src_items}</ul>
        </div>"""

    display(HTML(f"""
    <div style="font-family:Arial,sans-serif;max-width:920px;margin:10px auto;border-radius:8px;overflow:hidden;box-shadow:0 2px 8px rgba(0,0,0,0.1);">
      <div style="background:#1a252f;color:white;padding:12px 16px;">
        <strong>🔍 질문:</strong> {query}
      </div>
      <div style="display:grid;grid-template-columns:1fr 1fr;">
        <div style="padding:16px;background:#fdf2f2;border-right:2px solid #ddd;">
          <div style="color:#c0392b;font-weight:bold;font-size:14px;margin-bottom:10px;">❌ LLM 단독 (문서 없음)</div>
          <div style="font-size:13px;line-height:1.7;color:#555;">{llm_html}</div>
        </div>
        <div style="padding:16px;background:#f0fdf4;">
          <div style="color:#27ae60;font-weight:bold;font-size:14px;margin-bottom:10px;">✅ RAG (문서 검색 후)</div>
          <div style="font-size:13px;line-height:1.7;color:#222;">{rag_html}</div>
        </div>
      </div>
      {sources_html}
    </div>
    """))

    return rag_answer, retrieved

print("✅ 비교 함수 준비 완료")

In [ ]:
# ─────────────────────────────────────────
# 데모 1: 보안 정책 - LLM이 모르는 내부 정보
# ─────────────────────────────────────────
compare_llm_vs_rag(
    "테크코어의 API 키 유효기간은 얼마나 되나요?",
    mock_key="api_policy"
)

In [ ]:
# ─────────────────────────────────────────
# 데모 2: 릴리즈 노트 - 버전별 Breaking Change
# ─────────────────────────────────────────
compare_llm_vs_rag(
    "CloudSync v2.2에서 v2.3으로 업그레이드할 때 무엇을 확인해야 하나요?",
    mock_key="cloudsync_v23"
)

In [ ]:
# ─────────────────────────────────────────
# 데모 3: HR 정책 - 회사별 특수 정책
# ─────────────────────────────────────────
compare_llm_vs_rag(
    "재택근무 시 어떤 장비를 지원받을 수 있나요?",
    mock_key="wfh"
)

## 4️⃣ RAG 프롬프트 구조 분해하기

실제로 LLM에 들어가는 프롬프트가 어떻게 생겼는지 확인합니다.

In [ ]:
# RAG 프롬프트 내부 구조 시각화
query = "API 키 유효기간은?"
retrieved = vector_search(query, top_k=2)
rag_prompt = build_rag_prompt(query, retrieved)

print("=" * 55)
print("  RAG 프롬프트 내부 구조 (실제 LLM에 보내는 내용)")
print("=" * 55)
print()
print(f"총 길이: {len(rag_prompt)}자")
print(f"참고 문서 수: {len(retrieved)}개")
print()
print("─" * 55)
print(rag_prompt[:1500])
if len(rag_prompt) > 1500:
    print(f"... (이하 {len(rag_prompt)-1500}자 생략)")
print("─" * 55)
print()
print("💡 이 전체 텍스트가 LLM에 한 번에 전달됩니다.")
print("   LLM은 이 문서들을 '지금 이 질문을 위해 읽고' 답변합니다.")
print("   Fine-tuning과 달리 모델 가중치 변경이 없습니다!")

## 5️⃣ RAG의 한계 - 검색이 실패하면 답변도 실패

In [ ]:
# 검색 실패 → 답변 실패 시연

bad_query = "내년 인상률이 얼마나 될 것 같아?"  # 문서에 없는 내용
retrieved_bad = vector_search(bad_query, top_k=3)

print("=" * 55)
print(f"  문서에 없는 정보 검색 시도")
print(f"  쿼리: '{bad_query}'")
print("=" * 55)
print("\n검색된 문서 (관련성 낮음):")
for r in retrieved_bad:
    bar = "█" * int(r["score"] * 20)
    print(f"  [{r['doc_id']}] 유사도: {r['score']:.4f} {bar}")

print()
print("─" * 55)
print("LLM 답변 (올바른 RAG):")
mock_no_info = """죄송합니다. 제가 검색한 내부 문서들에서 급여 인상률에 대한 정보를
찾을 수 없습니다. 해당 내용은 HR팀 또는 직속 매니저에게 문의하시기 바랍니다.
[참고: 해당 정보 없음]"""

rag_prompt_bad = build_rag_prompt(bad_query, retrieved_bad)
answer = call_llm(rag_prompt_bad, mock=mock_no_info)
print(answer)

print()
print("🎯 핵심 포인트:")
print("   좋은 RAG 시스템은 '모른다'고 솔직하게 말합니다.")
print("   검색된 문서의 유사도가 낮으면 hallucination 위험 신호입니다.")
print("   threshold 설정: 유사도 0.3 이하면 '문서 없음'으로 처리하는 패턴 사용")

In [ ]:
# RAG 품질 요인 정리

display(HTML("""
<div style="font-family:Arial,sans-serif;max-width:750px;margin:10px auto;">
  <h3 style="color:#2c3e50;">RAG 답변 품질을 결정하는 요인들</h3>
  <table style="width:100%;border-collapse:collapse;font-size:13px;">
    <thead>
      <tr style="background:#2c3e50;color:white;">
        <th style="padding:10px;">요인</th>
        <th style="padding:10px;">잘못됐을 때</th>
        <th style="padding:10px;">해결 방향</th>
      </tr>
    </thead>
    <tbody>
      <tr style="background:#f8f9fa;">
        <td style="padding:10px;">🔍 검색 품질</td>
        <td style="padding:10px;">엉뚱한 문서 검색</td>
        <td style="padding:10px;">하이브리드 검색 (05번)</td>
      </tr>
      <tr>
        <td style="padding:10px;">✂️ 청킹 전략</td>
        <td style="padding:10px;">문맥이 잘린 조각 검색</td>
        <td style="padding:10px;">슬라이딩 윈도우, 문단 단위</td>
      </tr>
      <tr style="background:#f8f9fa;">
        <td style="padding:10px;">📊 문서 품질</td>
        <td style="padding:10px;">구버전/중복 문서 혼재</td>
        <td style="padding:10px;">문서 관리, 메타데이터 필터</td>
      </tr>
      <tr>
        <td style="padding:10px;">📝 프롬프트 설계</td>
        <td style="padding:10px;">LLM이 문서 무시하고 추측</td>
        <td style="padding:10px;">명확한 지침, 출처 요구</td>
      </tr>
      <tr style="background:#f8f9fa;">
        <td style="padding:10px;">🎯 top_k 설정</td>
        <td style="padding:10px;">너무 적으면 누락, 많으면 노이즈</td>
        <td style="padding:10px;">Reranking (05번)</td>
      </tr>
    </tbody>
  </table>
</div>
"""))

---

## 🎤 강의자 멘트 포인트

> **"RAG를 보여드릴 때 가장 드라마틱한 효과는 '내부 정보'입니다.**  
> **LLM 단독으로는 '잘 모르겠다'고 하던 것이,  
> RAG를 붙이면 '개발용 키는 90일, 운영용 키는 365일'이라고 정확하게 답합니다.**  
>
> 이게 모델 재학습 없이 일어난 일입니다.  
> Fine-tuning은 수백만 원, 몇 주가 걸리지만  
> RAG는 문서를 벡터로 바꾸고 검색만 연결하면 됩니다."

## 🙋 청중 질문 유도
> - "여러분 회사 내부 문서를 RAG에 연결한다면 어떤 문서부터 넣고 싶나요?"
> - "RAG 답변이 틀렸을 때 어떻게 디버깅할 수 있을까요?"
> - "RAG와 Fine-tuning을 각각 언제 쓰는 게 맞을까요?"

## 🏗️ 실무 확장 포인트
- **Auth**: 사용자 권한에 따라 검색 가능한 문서 제한
- **Citation**: 답변에 정확한 문서 링크 + 단락 번호 표시
- **Evaluation**: RAGAS 같은 프레임워크로 RAG 품질 자동 평가
- **Caching**: 자주 묻는 질문 + 검색 결과 캐싱으로 비용 절감
- **Feedback loop**: 사용자 피드백으로 검색 품질 개선

## ➕ 추가 실험 아이디어
1. `top_k`를 1, 3, 5로 바꿔서 답변 품질 비교
2. 구버전 SEC-POL-002만 넣고 RAG 실행해서 잘못된 답변 확인
3. 프롬프트에서 "문서에 없으면 솔직히 말하라" 지침을 제거하면 어떻게 되는지 확인

## ➡️ 다음 노트북
**05_better_rag_hybrid_and_rerank.ipynb** - naive RAG의 한계와 retrieval 품질 개선 방법